# Evaluating the text-to-SQL pipeline

This notebook measures how well the LangGraph agent in `Streamlit/agent.py` answers a bank of **39 questions**:

| | Olympic data (`OLYM`) | Human resources (`HR`) | Customer orders (`CO`) |
|---|---|---|---|
| Answerable questions | 10 | 10 | 10 |
| Honest-failure questions | 3 | 3 | 3 |

An *honest-failure* question has no answer in the data (for example a Winter Games in a Summer-only database). The right behavior is to say so, not to invent an answer.

## What "correct" means

- The truth for each answerable question is a **reference SQL**, written for this notebook. It runs against the same database, and a correct answer must contain every number and name in the first rows of its result.
- Step 4 checks each reference result against a fact known independently (for example 107 employees in HR), so a wrong reference SQL shows up before anything is scored.
- A pipeline answer counts as **correct** only if both hold: the sentence it writes contains the expected numbers and names, *and* the table it queried contains them too. Checking both stops a lucky word match from counting.
- An honest-failure question **passes** if the answer says it could not find the data or that the database does not have it.

## Metrics

| Metric | Definition |
|---|---|
| Accuracy | correct answers ÷ answerable questions |
| First-attempt success | correct, and the agent wrote its SQL only once |
| Retry rate | share of questions where the agent needed more than one attempt |
| Recovery rate | of the questions that needed a retry, the share that finished correct |
| Silently wrong | wrong, with no retry and no error: the agent was confident and mistaken |
| Honest-failure pass rate | questions declined correctly ÷ honest-failure questions |

## Read the results with care

- Keyword grading is blunt. An answer can be right but worded so the check misses it, or the reverse. Step 7 prints every failure so you can judge it yourself.
- 30 answerable questions is a small sample. The accuracy comes with a 95% confidence range (Wilson interval); quote the range, not only the percentage.
- Several questions were also used while the agent was being built (the 1984 USA golds, the woman with the most golds, the Soviet Union and Russia). Treat the score as a *development* score. For a held-out score, write new questions before looking at any answers.
- One run of the bank calls Gemini several times per question and queries FreeSQL. The agent spaces its Gemini calls to stay under the requests-per-minute limit in `MODEL_REQUESTS_PER_MINUTE` (15 for the current model, so one call every 4 seconds), which is why a run takes roughly 10 to 20 minutes. N_RUNS = 3 takes three times as long. A run can use over 100 requests, so a few runs a day can reach a 500-a-day limit; the agent then switches to its fallback model and prints a line saying so (scores from before and after the switch are from different models).

## 1. Setup

Connects to FreeSQL the same way the app does and builds the agent. Needs `GOOGLE_API_KEY`, `FREE_SQL_USER`, `FREE_SQL_PASSWORD` and `FREE_SQL_DSN` in your `.env`, as described in the README.

In [1]:
import os
import re
import sys
import time
from pathlib import Path

import oracledb
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import display

# GOOGLE_API_KEY, FREE_SQL_USER, FREE_SQL_PASSWORD, FREE_SQL_DSN. override=True makes the project's .env win over
# any older value left in the Windows environment, which would otherwise be used silently instead.
load_dotenv(find_dotenv(), override=True)

# Make "import agent" find Streamlit/agent.py, whether this runs from the project root or a subfolder.
ROOT = Path.cwd()
if not (ROOT / "Streamlit" / "agent.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "Streamlit"))
import importlib

import agent

agent = importlib.reload(agent)  # re-running this cell picks up edits to agent.py without restarting the kernel

N_RUNS = 1  # repeat the whole bank this many times; 3 shows how much answers vary from run to run
PAUSE = 0   # extra seconds between questions; the agent already keeps to the model's requests-per-minute limit
STOP_AFTER_ERRORS = 3  # stop the run if this many questions in a row crash, so a broken key or model does not waste minutes


def connect():
    return oracledb.connect(
        user=os.environ["FREE_SQL_USER"],
        password=os.environ["FREE_SQL_PASSWORD"],
        dsn=os.environ["FREE_SQL_DSN"],
    )


connection = connect()


def run_query(sql, params=None):
    """Run SQL on FreeSQL and return a DataFrame (the same job as run_query in app.py)."""
    global connection
    for attempt in (1, 2):
        try:
            with connection.cursor() as cur:
                cur.execute(sql, params or {})
                cols = [c[0] for c in cur.description]
                return pd.DataFrame(cur.fetchall(), columns=cols)
        except oracledb.Error as e:
            code = getattr(e.args[0], "full_code", "") if e.args else ""
            if attempt == 2 or code not in {"DPY-4011", "DPY-1001"}:
                raise
            connection = connect()  # the connection dropped during a long run: reconnect once


graph = agent.build_graph(run_query)
print(f"Model: {agent.MODEL_NAME} ({agent.MODELS[agent.MODEL_NAME]['requests_per_minute']} requests a minute), fallback if the daily limit is hit: {agent.FALLBACK_MODELS or 'none'}  |  retries allowed: {agent.MAX_RETRIES}  |  row limit: {agent.ROW_LIMIT}")

c:\gitProjects\LangStuff\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Model: minimax/minimax-m3.1-flash:free (20 requests a minute), fallback if the daily limit is hit: ['gemini-3.1-flash-lite', 'qwen/qwen3.8-27b', 'openai/gpt-oss-120b']  |  retries allowed: 3  |  row limit: 200


### Check the models before anything else

One tiny call to each model in `MODELS` (the main model, the fallbacks, and any others listed). If the main model fails here, the whole evaluation would fail, so this cell stops with the full error instead of letting 39 questions crash one by one. A failing fallback only prints a warning.

In [2]:
import hashlib

KEY_NAMES = {"google_genai": "GOOGLE_API_KEY", "groq": "GROQ_API_KEY", "openrouter": "OPENROUTER_API_KEY"}  # the .env key each provider uses
in_use = [agent.MODEL_NAME, *agent.FALLBACK_MODELS]  # the models the agent can really reach in this run

failures = {}
for name, config in agent.MODELS.items():
    key_name = KEY_NAMES.get(config["provider"], "?")
    key = os.environ.get(key_name, "")
    fingerprint = hashlib.sha256(key.encode()).hexdigest()[:8] if key else "NO KEY"  # a fingerprint, not the key
    role = "main" if name == agent.MODEL_NAME else "fallback" if name in in_use else "not used"
    started = time.perf_counter()
    try:
        reply = agent.make_client(name, agent.make_rate_limiter(name)).invoke("Reply with the single word: ok")
        print(f"ok      {name:<24} {role:<9} {key_name} {fingerprint}  {time.perf_counter() - started:4.1f}s  replied {reply.text.strip()[:20]!r}")
    except Exception as e:
        failures[name] = f"{type(e).__name__}: {str(e)[:300]}"
        print(f"FAILED  {name:<24} {role:<9} {key_name} {fingerprint}\n        {failures[name]}")

if agent.MODEL_NAME in failures:
    print("\nThe main model failed, so the evaluation cannot run until this is fixed.")
    print("Things to check: the key in .env (this notebook prefers it over the Windows environment),")
    print("that the key has not expired, and that the model name is in MODELS in Streamlit/agent.py.")
    raise RuntimeError(f"Model check failed for {agent.MODEL_NAME}: {failures[agent.MODEL_NAME]}")
for name in agent.FALLBACK_MODELS:
    if name in failures:
        print(f"\nWarning: the fallback model {name} failed. The main model works, but the switch on a daily limit would not.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


ok      gemini-3.5-flash-lite    not used  GOOGLE_API_KEY da283c67   5.7s  replied 'ok'
ok      gemini-3.1-flash-lite    fallback  GOOGLE_API_KEY da283c67  12.3s  replied 'ok'
ok      openai/gpt-oss-120b      fallback  GROQ_API_KEY 0a806640   2.7s  replied 'ok'
ok      qwen/qwen3.8-27b         fallback  GROQ_API_KEY 0a806640   2.2s  replied 'ok'
ok      minimax/minimax-m3.1-flash:free main      OPENROUTER_API_KEY 18ded4da   5.2s  replied 'ok'


## 2. The question bank

`known` is a fact I know independently of the database (host cities, official medal tables, the classic HR sample). It is only used in step 4, to catch a wrong reference SQL. `None` means there is no independent fact, so the database itself is the truth.

Olympic medal counts count *medals*, not athletes: a medal is one event at one Games (edition, discipline, event and event gender together), so a relay team's gold counts once.

In [3]:
MEDAL_KEY = "EDITION || '|' || DISCIPLINE || '|' || EVENT || '|' || EVENT_GENDER"
VIEW = "OLYM.OLYM_MEDALS_VIEW"


def golds(where):
    """Reference SQL: the number of distinct gold medals matching a condition."""
    return f"SELECT COUNT(DISTINCT {MEDAL_KEY}) AS GOLD_MEDALS FROM {VIEW} WHERE MEDAL = 'Gold' AND {where}"


def Q(id, schema, question, ref, known):
    """An answerable question: a reference SQL and, when I have one, an independent known fact."""
    return dict(id=id, schema=schema, kind="answerable", q=question, ref=ref, known=known)


def F(id, schema, question):
    """An honest-failure question: nothing in the data answers it."""
    return dict(id=id, schema=schema, kind="honest", q=question)


ORDERS_DONE = "UPPER(ORDER_STATUS) = 'COMPLETE'"

BANK = [
    # ---- Olympic data: summer Games 1896-2008 ----
    Q("O1", "OLYM", "Which city hosted the 1948 Olympics?",
      "SELECT CITY FROM OLYM.OLYM_GAMES WHERE YEAR = 1948", ["london"]),
    Q("O2", "OLYM", "How many Olympic Games are in the database?",
      "SELECT COUNT(DISTINCT YEAR) FROM OLYM.OLYM_GAMES", [26]),
    Q("O3", "OLYM", "In which years was the Olympics held in Los Angeles?",
      "SELECT DISTINCT YEAR FROM OLYM.OLYM_GAMES WHERE UPPER(CITY) LIKE '%LOS ANGELES%' ORDER BY YEAR", [1932, 1984]),
    Q("O4", "OLYM", "What is the most recent Olympic year in the database?",
      "SELECT MAX(YEAR) FROM OLYM.OLYM_GAMES", [2008]),
    Q("O5", "OLYM", "How many Olympic Games were held between 1900 and 1999?",
      "SELECT COUNT(DISTINCT YEAR) FROM OLYM.OLYM_GAMES WHERE YEAR BETWEEN 1900 AND 1999", [22]),
    Q("O6", "OLYM", "How many gold medals did the USA win at the 1984 Olympics?",
      golds("NOC = 'USA' AND EDITION = 1984"), [83]),
    Q("O7", "OLYM", "How many gold medals did the Soviet Union win at the 1980 Olympics?",
      golds("NOC = 'URS' AND EDITION = 1980"), [80]),
    Q("O8", "OLYM", "How many gold medals did Great Britain win at the 2008 Olympics?",
      golds("NOC = 'GBR' AND EDITION = 2008"), [19]),
    Q("O9", "OLYM", "How many gold medals did Michael Phelps win at the 2008 Olympics?",
      golds("UPPER(ATHLETE) LIKE 'PHELPS%' AND EDITION = 2008"), [8]),
    Q("O10", "OLYM", "Which woman has won the most gold medals, and how many?",
      f"SELECT ATHLETE, COUNT(DISTINCT {MEDAL_KEY}) AS GOLD_MEDALS FROM {VIEW} "
      "WHERE GENDER = 'Women' AND MEDAL = 'Gold' GROUP BY ATHLETE "
      "ORDER BY GOLD_MEDALS DESC FETCH FIRST 1 ROWS WITH TIES", ["latynina", 9]),

    # ---- Human resources: the classic Oracle HR sample ----
    Q("H1", "HR", "How many employees are there?",
      "SELECT COUNT(*) FROM HR.EMPLOYEES", [107]),
    Q("H2", "HR", "What is the highest salary?",
      "SELECT MAX(SALARY) FROM HR.EMPLOYEES", [24000]),
    Q("H3", "HR", "How many employees work in the Shipping department?",
      "SELECT COUNT(*) FROM HR.EMPLOYEES E JOIN HR.DEPARTMENTS D ON E.DEPARTMENT_ID = D.DEPARTMENT_ID "
      "WHERE D.DEPARTMENT_NAME = 'Shipping'", [45]),
    Q("H4", "HR", "How many departments are there?",
      "SELECT COUNT(*) FROM HR.DEPARTMENTS", [27]),
    Q("H5", "HR", "What is the average salary in the IT department?",
      "SELECT AVG(E.SALARY) FROM HR.EMPLOYEES E JOIN HR.DEPARTMENTS D ON E.DEPARTMENT_ID = D.DEPARTMENT_ID "
      "WHERE D.DEPARTMENT_NAME = 'IT'", [5760]),
    Q("H6", "HR", "Which department has the most employees, and how many?",
      "SELECT D.DEPARTMENT_NAME, COUNT(*) AS EMPLOYEES FROM HR.EMPLOYEES E "
      "JOIN HR.DEPARTMENTS D ON E.DEPARTMENT_ID = D.DEPARTMENT_ID "
      "GROUP BY D.DEPARTMENT_NAME ORDER BY EMPLOYEES DESC FETCH FIRST 1 ROWS WITH TIES", ["shipping", 45]),
    Q("H7", "HR", "How many employees have no department?",
      "SELECT COUNT(*) FROM HR.EMPLOYEES WHERE DEPARTMENT_ID IS NULL", [1]),
    Q("H8", "HR", "How many different jobs are there?",
      "SELECT COUNT(*) FROM HR.JOBS", [19]),
    Q("H9", "HR", "Which employees earn more than their manager?",
      "SELECT E.FIRST_NAME, E.LAST_NAME FROM HR.EMPLOYEES E JOIN HR.EMPLOYEES M ON E.MANAGER_ID = M.EMPLOYEE_ID "
      "WHERE E.SALARY > M.SALARY ORDER BY E.LAST_NAME", ["ozer", "abel"]),
    Q("H10", "HR", "Which department has the highest average salary, and what is it?",
      "SELECT D.DEPARTMENT_NAME, ROUND(AVG(E.SALARY), 2) AS AVG_SALARY FROM HR.EMPLOYEES E "
      "JOIN HR.DEPARTMENTS D ON E.DEPARTMENT_ID = D.DEPARTMENT_ID "
      "GROUP BY D.DEPARTMENT_NAME ORDER BY AVG_SALARY DESC FETCH FIRST 1 ROWS WITH TIES", ["executive", 19333.33]),

    # ---- Customer orders (sales and order counts use COMPLETE orders only, unless a status is named) ----
    Q("C1", "CO", "How many customers are there?",
      "SELECT COUNT(*) FROM CO.CUSTOMERS", [392]),
    Q("C2", "CO", "How many stores are there?",
      "SELECT COUNT(*) FROM CO.STORES", [23]),
    Q("C3", "CO", "How many products are there?",
      "SELECT COUNT(*) FROM CO.PRODUCTS", [46]),
    Q("C4", "CO", "How many orders are there in total, including cancelled and refunded ones?",
      "SELECT COUNT(*) FROM CO.ORDERS", [1950]),
    Q("C5", "CO", "How many completed orders are there?",
      f"SELECT COUNT(*) FROM CO.ORDERS WHERE {ORDERS_DONE}", [1892]),
    Q("C6", "CO", "How many orders were cancelled?",
      "SELECT COUNT(*) FROM CO.ORDERS WHERE UPPER(ORDER_STATUS) = 'CANCELLED'", [35]),
    Q("C7", "CO", "How many orders were refunded?",
      "SELECT COUNT(*) FROM CO.ORDERS WHERE UPPER(ORDER_STATUS) = 'REFUNDED'", [23]),
    Q("C8", "CO", "What are the five best-selling products by revenue?",
      "SELECT P.PRODUCT_NAME, SUM(OI.UNIT_PRICE * OI.QUANTITY) AS REVENUE FROM CO.ORDER_ITEMS OI "
      "JOIN CO.ORDERS O ON O.ORDER_ID = OI.ORDER_ID JOIN CO.PRODUCTS P ON P.PRODUCT_ID = OI.PRODUCT_ID "
      "WHERE UPPER(O.ORDER_STATUS) = 'COMPLETE' GROUP BY P.PRODUCT_NAME "
      "ORDER BY REVENUE DESC FETCH FIRST 5 ROWS ONLY", None),
    Q("C9", "CO", "Which customer has placed the most orders, and how many?",
      "SELECT C.FULL_NAME, COUNT(*) AS ORDERS FROM CO.ORDERS O JOIN CO.CUSTOMERS C ON C.CUSTOMER_ID = O.CUSTOMER_ID "
      "WHERE UPPER(O.ORDER_STATUS) = 'COMPLETE' GROUP BY C.FULL_NAME "
      "ORDER BY ORDERS DESC FETCH FIRST 1 ROWS WITH TIES", None),
    Q("C10", "CO", "What is the total revenue from completed orders?",
      "SELECT SUM(OI.UNIT_PRICE * OI.QUANTITY) FROM CO.ORDER_ITEMS OI JOIN CO.ORDERS O ON O.ORDER_ID = OI.ORDER_ID "
      "WHERE UPPER(O.ORDER_STATUS) = 'COMPLETE'", None),

    # ---- Honest failures: the right answer is "the data does not have this" ----
    F("OF1", "OLYM", "How many gold medals did the USA win at the 1980 Winter Olympics?"),   # Summer Games only
    F("OF2", "OLYM", "Who won the most gold medals at the 2024 Paris Olympics?"),             # data ends in 2008
    F("OF3", "OLYM", "How many medals did Russia win at the 1984 Olympics?"),                 # the USSR boycotted 1984
    F("HF1", "HR", "What is Steven King's home address?"),                                    # no home addresses stored
    F("HF2", "HR", "What is the average performance rating of employees?"),                   # no ratings stored
    F("HF3", "HR", "Which employees are on vacation right now?"),                             # no leave data
    F("CF1", "CO", "What is the profit margin on each product?"),                             # no cost data
    F("CF2", "CO", "What is the average customer satisfaction rating?"),                      # no ratings
    F("CF3", "CO", "How many orders were placed in 2030?"),                                   # no orders that late
]

answerable = [q for q in BANK if q["kind"] == "answerable"]
honest = [q for q in BANK if q["kind"] == "honest"]
assert len(answerable) == 30 and len(honest) == 9, (len(answerable), len(honest))
print(f"{len(BANK)} questions: {len(answerable)} answerable and {len(honest)} honest-failure")
display(pd.DataFrame(BANK).groupby(["schema", "kind"]).size().unstack())

39 questions: 30 answerable and 9 honest-failure


kind,answerable,honest
schema,,
CO,10,3
HR,10,3
OLYM,10,3


## 3. How answers are graded

Small functions, so the rules are easy to read and change. A reference result becomes a list of *tokens* (numbers and lower-case names). Numbers match within a small tolerance, so `19,333.33`, `19333.33` and `19,333` all match 19333.33.

In [4]:
NUMBER = re.compile(r"\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?")

# Phrases that mean "I could not find that in the data" (checked in lower case, with plain apostrophes).
HEDGES = (
    "couldn't find", "could not find", "cannot find", "can't find", "no matching", "does not contain",
    "doesn't contain", "do not contain", "not in the database", "not available", "no data", "no record",
    "not found", "unable to", "no information", "there is no", "there are no", "does not have",
    "doesn't have", "do not have", "not stored", "not recorded", "does not hold", "do not hold",
    "does not provide", "cannot be determined", "cannot be calculated", "not tracked",
    "no results", "no rows", "0 matching", "zero matching", "found nothing", "found 0", "returned no",
)

def tidy(text):
    """Plain form of an answer, so odd characters do not break the matching.

    Some models write 15 520.64 with a narrow no-break space (U+202F) as the thousands
    separator, put the same space inside names, or use curly quotes and non-breaking hyphens.
    """
    text = re.sub(r"(?<=\d)[\u202f\u00a0\u2009](?=\d{3}\b)", "", str(text))  # 15 520.64 -> 15520.64
    text = re.sub(r"[\u202f\u00a0\u2009\u2007]", " ", text)  # any other special space -> a normal space
    for odd, plain in {"\u2011": "-", "\u2010": "-", "\u2013": "-", "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"'}.items():
        text = text.replace(odd, plain)
    text = re.sub(r"[*`]", "", text)  # markdown bold and code marks, so **0** reads as 0
    return text

def numbers_in(text):
    """Every number in a piece of text, with thousands separators removed."""
    return [float(m.group(0).replace(",", "")) for m in NUMBER.finditer(tidy(text))]


def close(a, b):
    """Two numbers match if they agree to within 0.01, or 0.05% for large values."""
    return abs(a - b) <= max(0.01, 5e-4 * abs(b))


def tokens_from_df(df, rows=5):
    """What a correct answer must contain: each number and name in the first `rows` rows of a result."""
    tokens = []
    for value in df.head(rows).to_numpy().ravel():
        if value is None or (isinstance(value, float) and pd.isna(value)):
            continue
        try:
            tokens.append(round(float(value), 2))  # numbers (int, float, Decimal)
        except (TypeError, ValueError):
            text = str(value).strip()
            tokens.append((text.split(",")[0] if ", " in text else text).lower())  # "LATYNINA, Larisa" -> "latynina"
    return list(dict.fromkeys(tokens))  # drop repeats, keep order


def answer_has(tokens, text):
    """True if the sentence contains every expected number (within tolerance) and name."""
    nums, low = numbers_in(text), tidy(text).lower()
    return all((any(close(n, t) for n in nums) if isinstance(t, float) else t in low) for t in tokens)


def frame_has(tokens, df):
    """True if the table contains every expected number (within tolerance) and name."""
    if df is None or len(df) == 0:
        return False
    nums, texts = [], []
    for cell in df.to_numpy().ravel():
        try:
            nums.append(float(cell))
        except (TypeError, ValueError):
            texts.append(str(cell).lower())
    return all((any(close(n, t) for n in nums) if isinstance(t, float) else any(t in s for s in texts)) for t in tokens)


def declines(text):
    """True if the answer says it could not find the data (the pass condition for honest-failure questions)."""
    low = tidy(text).lower()
    return any(phrase in low for phrase in HEDGES)


def wilson(k, n, z=1.96):
    """95% confidence range for k successes out of n (the Wilson interval)."""
    if n == 0:
        return float("nan"), float("nan")
    p, d = k / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / d
    return centre - half, centre + half

## 4. Check the question bank first

Runs every reference SQL (no model calls, so it is quick and free) and compares each result with the known fact. Look at anything marked `DIFFERS` or `ERROR` before trusting a score:

- `ERROR`: the reference SQL failed (a wrong column name, for example). Fix the SQL in the bank.
- `DIFFERS`: the reference result is not the known fact. Either the SQL is wrong, or the dataset genuinely differs from official numbers (Olympic medal counts are the likeliest). The evaluation grades against the database, so a `DIFFERS` question is still scored fairly; it just means my known fact should be corrected.

In [5]:
checks = []
for q in answerable:
    try:
        q["ref_df"] = run_query(q["ref"])
        q["tokens"] = tokens_from_df(q["ref_df"])
        if not q["known"]:
            status = "no known fact (the database is the truth)"
        else:
            known = [float(k) if isinstance(k, (int, float)) else k for k in q["known"]]
            status = "matches known fact" if frame_has(known, q["ref_df"]) else "DIFFERS from known fact"
        shown = " | ".join(", ".join(str(v) for v in row) for row in q["ref_df"].head(3).to_numpy().tolist()) or "(no rows)"
    except Exception as e:
        q["ref_df"], q["tokens"] = None, None
        status, shown = f"ERROR: {str(e)[:90]}", ""
    checks.append(dict(id=q["id"], question=q["q"], known=q["known"], reference_result=shown, status=status))

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
check_table = pd.DataFrame(checks)
display(check_table)
print(check_table.status.str.split(":").str[0].value_counts().to_string())

,id,question,known,reference_result,status
0,O1,Which city hosted the 1948 Olympics?,[london],London,matches known fact
1,O2,How many Olympic Games are in the database?,[26],26,matches known fact
2,O3,In which years was the Olympics held in Los Angeles?,"[1932, 1984]",1932 | 1984,matches known fact
3,O4,What is the most recent Olympic year in the database?,[2008],2008,matches known fact
4,O5,How many Olympic Games were held between 1900 and 1999?,[22],22,matches known fact
5,O6,How many gold medals did the USA win at the 1984 Olympics?,[83],82,DIFFERS from known fact
6,O7,How many gold medals did the Soviet Union win at the 1980 Olympics?,[80],80,matches known fact
7,O8,How many gold medals did Great Britain win at the 2008 Olympics?,[19],19,matches known fact
8,O9,How many gold medals did Michael Phelps win at the 2008 Olympics?,[8],8,matches known fact
9,O10,"Which woman has won the most gold medals, and how many?","[latynina, 9]","LATYNINA, Larisa, 9",matches known fact


status
matches known fact                           26
no known fact (the database is the truth)     3
DIFFERS from known fact                       1


## 5. Run the pipeline

Asks every question through the real graph, the way the app does, and records the answer, the SQL, how many attempts it took, and how long. A question whose reference SQL failed in step 4 is skipped and listed.

In [6]:
todo = [q for q in BANK if q["kind"] == "honest" or q.get("tokens") is not None]
skipped = [q["id"] for q in BANK if q not in todo]
if skipped:
    print("Skipped (fix their reference SQL first):", skipped)
tokens_by_id = {q["id"]: q.get("tokens") for q in BANK}
print(f"Running {len(todo)} questions x {N_RUNS} run(s)\n")

records, crashes_in_a_row = [], 0
for run in range(1, N_RUNS + 1):
    for q in todo:
        agent.MODELS_USED.clear()  # so only this question's calls are counted
        started = time.perf_counter()
        try:
            final, failure = graph.invoke({"question": q["q"], "schema": q["schema"]}), ""
            crashes_in_a_row = 0
        except Exception as e:  # a crash is a failed question, not a stopped evaluation...
            final, failure = {}, f"{type(e).__name__}: {e}"
            crashes_in_a_row += 1
            if crashes_in_a_row >= STOP_AFTER_ERRORS:  # ...unless it keeps happening, which means something is broken
                raise RuntimeError(f"{crashes_in_a_row} questions in a row crashed, so the run was stopped. The last error was:\n{failure}") from e
        seconds = time.perf_counter() - started
        models = ", ".join(dict.fromkeys(agent.MODELS_USED)) or "none"  # in order; two names mean it switched models mid-question

        answer = str(final.get("answer", ""))
        record = dict(
            id=q["id"], schema=q["schema"], kind=q["kind"], run=run, question=q["q"],
            model=models, attempts=final.get("attempts", 0), seconds=round(seconds, 1),
            error=failure or final.get("error", ""), sql=final.get("sql", ""), answer=answer,
        )
        if q["kind"] == "answerable":
            record["answer_ok"] = answer_has(q["tokens"], answer)
            record["data_ok"] = frame_has(q["tokens"], final.get("df"))
            record["correct"] = record["answer_ok"] and record["data_ok"]
        else:
            record["answer_ok"] = record["data_ok"] = None
            record["correct"] = declines(answer)
        records.append(record)
        print(f"run {run}  {q['id']:<4} {'PASS' if record['correct'] else 'FAIL'}  attempts={record['attempts']}  {seconds:5.1f}s  {models}  {q['q'][:50]}")
        time.sleep(PAUSE)

results = pd.DataFrame(records)
print(f"\nDone: {int(results.correct.sum())} of {len(results)} passed")

Running 39 questions x 1 run(s)

run 1  O1   PASS  attempts=1    8.0s  minimax/minimax-m3.1-flash:free  Which city hosted the 1948 Olympics?
run 1  O2   PASS  attempts=1    5.0s  minimax/minimax-m3.1-flash:free  How many Olympic Games are in the database?
run 1  O3   PASS  attempts=1    6.4s  minimax/minimax-m3.1-flash:free  In which years was the Olympics held in Los Angele
run 1  O4   PASS  attempts=1    7.0s  minimax/minimax-m3.1-flash:free  What is the most recent Olympic year in the databa
run 1  O5   PASS  attempts=1    7.6s  minimax/minimax-m3.1-flash:free  How many Olympic Games were held between 1900 and 
run 1  O6   PASS  attempts=1    6.9s  minimax/minimax-m3.1-flash:free  How many gold medals did the USA win at the 1984 O
run 1  O7   PASS  attempts=1   16.4s  minimax/minimax-m3.1-flash:free  How many gold medals did the Soviet Union win at t
run 1  O8   PASS  attempts=1    5.8s  minimax/minimax-m3.1-flash:free  How many gold medals did Great Britain win at the 
run 1  O9   

## 6. Results

Accuracy, first-attempt success and recovery rate are computed on the answerable questions only; honest failures are reported separately, because "first attempt" does not mean much for a question that has no answer.

In [7]:
ans = results[results.kind == "answerable"].copy()
hon = results[results.kind == "honest"].copy()
ans["first_try"] = ans.correct & (ans.attempts == 1)
ans["retried"] = ans.attempts > 1
ans["silent_wrong"] = ~ans.correct & ~ans.retried & (ans.error == "")
retried = ans[ans.retried]


def frac(k, n):
    return f"{int(k)}/{n} = {k / n:.0%}" if n else "n/a"


n, k = len(ans), int(ans.correct.sum())
low, high = wilson(k, n)
summary = pd.DataFrame(
    [
        ["Accuracy", frac(k, n), f"95% confidence range {low:.0%} to {high:.0%}"],
        ["First-attempt success", frac(ans.first_try.sum(), n), "correct, with the SQL written once"],
        ["Retry rate", frac(ans.retried.sum(), n), "needed more than one attempt"],
        ["Recovery rate", frac(retried.correct.sum(), len(retried)), "of the retried questions, finished correct"],
        ["Silently wrong", frac(ans.silent_wrong.sum(), n), "wrong, no retry, no error"],
        ["Honest-failure pass rate", frac(hon.correct.sum(), len(hon)), "said the data does not have it"],
        ["Time per question", f"median {results.seconds.median():.0f}s, 95th percentile {results.seconds.quantile(0.95):.0f}s", "includes retries"],
    ],
    columns=["Metric", "Result", "Notes"],
)
display(summary.set_index("Metric"))

rows = []
for schema, g in ans.groupby("schema"):
    r = g[g.retried]
    h = hon[hon.schema == schema]
    rows.append(dict(
        schema=schema, questions=len(g), accuracy=f"{g.correct.mean():.0%}", first_attempt=f"{g.first_try.mean():.0%}",
        retry_rate=f"{g.retried.mean():.0%}", recovery=f"{r.correct.mean():.0%}" if len(r) else "n/a",
        honest_failures=f"{int(h.correct.sum())}/{len(h)}",
    ))
print("\nBy schema")
display(pd.DataFrame(rows).set_index("schema"))

print("Attempts per answerable question")
display(ans.attempts.value_counts().sort_index().rename("questions").to_frame())

if N_RUNS > 1:
    print("Accuracy for each run")
    display(ans.groupby("run").correct.mean().map("{:.0%}".format).rename("accuracy").to_frame())

print("Data covered by the three schemas")
display(run_query(
    "SELECT owner AS SCHEMA, COUNT(DISTINCT table_name) AS TABLES, COUNT(*) AS COLUMNS "
    "FROM all_tab_columns WHERE owner IN ('OLYM', 'HR', 'CO') GROUP BY owner ORDER BY owner"
))

,Result,Notes
Metric,,
Accuracy,30/30 = 100%,95% confidence range 89% to 100%
First-attempt success,28/30 = 93%,"correct, with the SQL written once"
Retry rate,2/30 = 7%,needed more than one attempt
Recovery rate,2/2 = 100%,"of the retried questions, finished correct"
Silently wrong,0/30 = 0%,"wrong, no retry, no error"
Honest-failure pass rate,9/9 = 100%,said the data does not have it
Time per question,"median 6s, 95th percentile 25s",includes retries



By schema


,questions,accuracy,first_attempt,retry_rate,recovery,honest_failures
schema,,,,,,
CO,10,100%,100%,0%,n/a,3/3
HR,10,100%,100%,0%,n/a,3/3
OLYM,10,100%,80%,20%,100%,3/3


Attempts per answerable question


,questions
attempts,
1,28
2,2


Data covered by the three schemas


,SCHEMA,TABLES,COLUMNS
0,CO,7,43
1,HR,7,35
2,OLYM,10,38


## 7. Review every failure

The grading is keyword-based, so read these. If an answer is actually right but worded in a way the check missed, that is a grading limit, not a pipeline failure; note it when you quote the numbers.

In [8]:
failed = results[~results.correct]
print(f"{len(failed)} of {len(results)} runs failed\n")
for _, r in failed.iterrows():
    print(f"[{r.id}] {r.question}")
    print(f"  attempts: {r.attempts}    error: {(r.error or '-')[:400]}")
    print(f"  SQL: {(r.sql or '-')[:320]}")
    print(f"  answer: {r.answer[:320]}")
    if r.kind == "answerable":
        print(f"  expected: {tokens_by_id[r.id]}   (answer had them: {r.answer_ok}, table had them: {r.data_ok})")
    else:
        print("  expected: an answer saying the data does not have this")
    print()

0 of 39 runs failed



## 8. Save the results

Writes every answer and score to `eval_results.csv` (not committed unless you choose to) and prints a table you can paste into the README.

In [9]:
out = ROOT / "eval_results.csv"
results.to_csv(out, index=False)
print(f"Saved {len(results)} rows to {out}\n")

print(f"Paste into the README ({n} answerable and {len(hon)} honest-failure questions, {N_RUNS} run(s), model {agent.MODEL_NAME}):\n")
print("| Metric | Result |")
print("|---|---|")
for _, row in summary.iterrows():
    print(f"| {row['Metric']} | {row['Result']} ({row['Notes']}) |")

Saved 39 rows to c:\gitProjects\LangStuff\eval_results.csv

Paste into the README (30 answerable and 9 honest-failure questions, 1 run(s), model minimax/minimax-m3.1-flash:free):

| Metric | Result |
|---|---|
| Accuracy | 30/30 = 100% (95% confidence range 89% to 100%) |
| First-attempt success | 28/30 = 93% (correct, with the SQL written once) |
| Retry rate | 2/30 = 7% (needed more than one attempt) |
| Recovery rate | 2/2 = 100% (of the retried questions, finished correct) |
| Silently wrong | 0/30 = 0% (wrong, no retry, no error) |
| Honest-failure pass rate | 9/9 = 100% (said the data does not have it) |
| Time per question | median 6s, 95th percentile 25s (includes retries) |
